# 4 · From a Sentinel-2 image to a land-cover map

In [notebook 3](03_inputs.ipynb) we wrote `LandCoverPredictor` to turn one prepared image into per-class logits. Now start with one of the same validation GeoTIFFs and turn the whole observation into a class map with one call.

```text
raw image → prepare bands → predict and blend chip logits → decode land-cover classes
```

Use a fresh kernel in `examples/` after running notebook 3. The saved pipeline can then map the KHDTK forest using the search from [notebook 2](02_sources.ipynb). That search needs internet access.

## Restore the training checkpoint

The checkpoint holds the encoder, decoder and head recipe alongside their trained weights. Lightning prefixes those weights with `model.` because the chain was stored in `Segmenter.model`.

In [ ]:
from pathlib import Path

import torch
import xarray as xr
import yaml

import geosave as gs
from geosave.ml import RasterPipeline, build_model

checkpoint = torch.load("data/checkpoints/03_inputs/last.ckpt", map_location="cpu", weights_only=True)
model = build_model(checkpoint["stages"]).eval()
weights = {
    name.removeprefix("model."): tensor
    for name, tensor in checkpoint["state_dict"].items()
    if name.startswith("model.")
}
model.load_state_dict(weights)

## Reuse the predictor

Run the predictor definition from notebook 3 in this fresh kernel. The checkpoint supplies the trained weights.

In [ ]:
from dataclasses import dataclass
from itertools import batched

from torch import nn
from torch.utils.data import default_collate

from geosave.ml import chain_step
from geosave.ml.processing.raster.prediction.base import RasterPredictor


@dataclass
class LandCoverPredictor(RasterPredictor):
    size: int = 128
    stride: int = 64
    batch_size: int = 16

    def measure_chips(self) -> tuple[int, int]:
        return self.size, self.stride

    @chain_step()
    def __call__(self, model: nn.Module, sample: xr.DataTree) -> xr.DataArray:
        chips = gs.RasterChipper([sample], size=self.size, stride=self.stride)
        merger = gs.RasterMerger(chips.layout, window="hann")
        device = next(model.parameters()).device

        for chip_indices in batched(range(len(chips)), self.batch_size):
            chip_inputs = []
            for index in chip_indices:
                pixels, context = chips[index].gs.to_tensor(context=["centroid", "year", "day_of_year"])
                chip_inputs.append({**pixels, **context})

            batch = default_collate(chip_inputs)
            inputs = {name: tensor.to(device) for name, tensor in batch.items()}
            with torch.inference_mode():
                batch_logits = model(**inputs).float().cpu().numpy()
            for index, chip_logits in zip(chip_indices, batch_logits, strict=True):
                merger.add(index, chip_logits)

        output = merger.merge()
        return output

## Assemble the three steps

Use the predictor from notebook 3. Give it the restored model and the same four bands as before. The preprocessor decodes reflectance and fills missing pixels with zero. The predictor merges chip logits. The postprocessor takes the largest logit at each pixel and attaches the training legend.

In [ ]:
from geosave.ml.processing.raster.preprocessing.dense import DensePreprocessor
from geosave.ml.processing.raster.postprocessing.dense import SegmentationPostprocessor

root = Path("data/dynamic_world")
target = yaml.safe_load((root / "labels.yaml").read_text())["dynamic_world"]
BANDS = ["B02", "B03", "B04", "B08"]

pipeline = RasterPipeline(
    model=model,
    preprocessor=DensePreprocessor(bands=BANDS),
    predictor=LandCoverPredictor(),
    postprocessor=SegmentationPostprocessor(
        classes=target["classes"], colors=target["colors"]
    ),
)

## Call the pipeline on one validation image

Open the stored bands as they are. `pipeline.predict(rasters)` runs all three steps and returns the land-cover map on the image grid.

In [ ]:
val_images = sorted((root / "image" / "val").glob("*.tif"))
scene = gs.open_raster(val_images[0])
rasters = xr.DataTree.from_dict({"image": scene})
cover = pipeline.predict(rasters)
cover["class_id"]

In [ ]:
scene.gs.plot(rgb=("B04", "B03", "B02"), title="Validation image") + cover[
    "class_id"
].gs.plot(title="Pipeline prediction")

## Save and reload the pipeline

Save the weights together with the band preparation, chip settings and class legend. Run the same image through the reloaded pipeline and compare its map with the one above. The config refers to `LandCoverPredictor` by its Python class name. When reloading in a fresh notebook kernel, run the predictor definition above first.

In [ ]:
from geosave.ml.release import ChainModel, load_pipeline

release = Path("data/landcover_pipeline")
ChainModel.from_pipeline(pipeline).save_pretrained(release)

reloaded = load_pipeline(release)
reloaded.predict(rasters).equals(cover)

## Load forest observations

Apply the region search from [notebook 2](02_sources.ipynb) to the KHDTK forest boundary. Select one low-cloud Sentinel-2 Item per month and load the six observations onto a common 10-metre grid.

In [ ]:
from odc.geo.geobox import GeoBox

client = gs.StacClient.planetary_computer()
packing = gs.attrs.CFPacking(scale_factor=0.0001, fill_value=0)
boundary = gs.open_vector("khdtk.geojson")
forest = gs.GeoRegion.from_geometry(boundary)
forest_grid = GeoBox.from_geopolygon(forest.extent, crs="EPSG:32749", resolution=10)
forest_query = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=1)
forest_query = forest_query.sort_by("properties.eo:cloud_cover")

forest_items = []
for month in ["2019-05", "2019-06", "2019-07", "2019-08", "2019-09", "2019-10"]:
    matches = client.search_items(forest_query.set_region(forest.set_time(month)))
    if not matches:
        raise ValueError(f"No forest observation found for {month}")
    forest_items.extend(matches)

scenes = gs.stac.load_dataset(
    forest_items, bands=BANDS, geobox=forest_grid, groupby="solar_day",
    dtype="uint16", nodata=0, patch_url=client.patch_url, chunks={},
).gs.write_packing(packing)
scenes

## Publish the maps

A prediction is a raster like any other. Each map keeps its observation's time, so its Item is dated like the imagery it was predicted from.

In [ ]:
predictions = Path("data/khdtk_maps")

items = []
for position in range(scenes.sizes["time"]):
    scene = scenes.isel(time=position)
    cover = reloaded.predict(xr.DataTree.from_dict({"image": scene}))
    day = scene.gs.timespan[0].strftime("%Y%m%d")
    items.append(cover.gs.publish_item(predictions, id=day, overwrite=True))

catalog = gs.stac.create_catalog(items)
catalog.gs.to_geoparquet(predictions / "catalog.parquet", overwrite=True)
catalog[["id", "datetime"]]

### Optional: open the maps in the catalog viewer

```python
explorer = catalog.gs.explore()
server = explorer.serve(threaded=True, open=False)
# When finished:
explorer.stop()
```

## Explore further

### Share the pipeline

Share this notebook alongside the saved release. Its predictor cell defines the class that `load_pipeline` rebuilds when loading the weights and configuration. See [Release and run a model](../docs/advanced/release.md).

### Map a larger area

`pipeline.predict_to_zarr(rasters, "forest.zarr")` predicts and writes one chunk at a time. Open the store and publish it like any other product.

### Keep a reference case

Save one observation and its expected map with each release to catch changes in preparation or dependencies.